## 3 кейс

**В этом кейсе вы будете рассчитывать:**
* retention
* rolling retention
* lifetime
* churn rate
* mau
* wau
* dau

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [1]:
!wget https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv

!wget https://gist.github.com/Vs8th/aacb80595d1d6aaa2e31eb735f8bc644/raw/entries.csv

!wget https://gist.github.com/Vs8th/0e827e9a608117345dd6585ab81e8c86/raw/metrics.txt

--2026-03-01 13:04:28--  https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.github.com (gist.github.com)... 140.82.113.3
Connecting to gist.github.com (gist.github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv [following]
--2026-03-01 13:04:28--  https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14918 (15K) [text/plain]
Saving to: ‘registrations.csv’

registrations.csv   100%[===================>]  14.57K  --.-KB/s    in 0s      

2026-03-01 13:04:28 (87.1 M

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Файлами для работы являются `registrations.csv` и `entries.csv`. В них хранятся данные о регистрациях пользователей и входа на платформу соответственно.

### **Посчитайте Retention 15 дня (в процентах) для пользователей, зарегистрированных в январе**

Cохраните результат в переменную `retention_15_day`

**Примечание:** результат округлите до 5 знаков после запятой

In [3]:
# Ваше решение
df = pd.read_csv('registrations.csv', sep =';', parse_dates = ['registration_date'])
df1 = pd.read_csv('entries.csv', sep =';', parse_dates = ['entry_date'])

In [4]:
january_df = df[df['registration_date'].dt.month == 1]

In [5]:
january_df1 = january_df.merge(df1, on = 'user_id', how='inner')

In [6]:
january_df1['days_since_registration'] = (january_df1['entry_date'] - january_df1['registration_date']).dt.days

In [7]:
cohort_size = january_df1['user_id'].nunique()
returned_15_day = january_df1[january_df1['days_since_registration'] == 15]['user_id'].nunique()
retention_15_day = round(returned_15_day / cohort_size*100, 5)

In [8]:
retention_15_day

54.65116

In [9]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
# Открываем файл с правильными ответами
with open('metrics.txt', 'r') as f:
    answers = f.read().split('\n')

correct_answer = float(answers[0])

try:
    assert retention_15_day == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Rolling-retention 30 дня (в процентах) для пользователей из той же когорты**

Сохраните результат в переменную `rolling_retention`

**Примечание:** результат округлите до 5 знаков после запятой

In [10]:
from pandas.core.window import rolling
# Ваше решение
last_activity = january_df1.groupby('user_id')['days_since_registration'].max().reset_index()
rolling_df = january_df1.merge(last_activity, on = 'user_id', how='left')
rolling_30 = rolling_df[rolling_df['days_since_registration_x'] >=30]['user_id'].nunique()
rolling_retention = round(rolling_30 / cohort_size*100, 5)
rolling_retention

29.06977

In [11]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[1])

try:
    assert rolling_retention == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Lifetime по всем пользователям, посчитанный как интеграл от n-day retention**

Сохраните результат в переменную `lifetime`

**Примечание:** результат округлите до 5 знаков после запятой

In [30]:
# Ваше решение
#нужно объединить два df
#удалить повторные дубликаты входа
#посчитать разницу в днях между датой регистрации и датой захода date_range
#сгруппировать новый df по date_range и количеству пользователей (уникальных пользователей)
#посчитать долю путем деления количества пользователей в n-day на количество пользователей в 0-day
#просуммировать эти доли и сравнить с результатом в файле metrics
new_df = df.merge(df1, on = 'user_id', how = 'left')
new_df['date_diff'] = (new_df['entry_date'] - new_df['registration_date']).dt.days
new_df_1 = new_df.groupby('date_diff')['user_id'].nunique().reset_index()
new_df_1['ret_part'] = new_df_1['user_id']/new_df_1['user_id'].iloc[0]
lifetime = round(new_df_1['ret_part'].sum(), 5)
lifetime

np.float64(14.804)

In [31]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[2])

try:
    assert lifetime == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Churn rate 29 дня (в долях), посчитанный по всем пользователям**

Сохраните результат в переменную `churn_29`.  
Если будете считать CR от Retention, ведите расчет от Rolling Retention, таким образом, мы получим всех, кто не заходил в 29 день и после. Ведя расчет от обычного Retention, наоборот - получим только CR в 29 день.


In [ ]:
# Ваше решение



In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[3])

try:
    assert churn_29 == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

### **Посчитайте Mau, Wau, Dau за последний месяц/неделю/день записей**

Сохраните результат в переменные `dec_mau`, `dec_wau`, `dec_dau` соответственно

**Примечание:** последний месяц записей - декабрь. Поэтому `mau` рассчитываем для декабря (2021 года), для `wau` берем последнюю неделю - с 25 по 31 декабря, и для `dau` соответственно последний день - 31 декабря.

In [39]:
# Ваше решение
new_df.head()
dec_df = new_df[new_df['entry_date'].dt.month == 12]
dec_mau = dec_df['user_id'].nunique()
dec_mau
dec_wau = dec_df[(dec_df['entry_date'] >= '2021-12-25') & (dec_df['entry_date'] <= '2021-12-31')]['user_id'].nunique()
dec_wau
dec_dau = dec_df[dec_df['entry_date'] == '2021-12-31']['user_id'].nunique()
dec_dau

47

In [49]:
avg_dau = new_df.groupby('entry_date')['user_id'].nunique().mean()
avg_dau = avg_dau.round(5)

np.float64(40.5589)

In [54]:
new_df['month'] = new_df['entry_date'].dt.month
avg_mau = new_df.groupby('month')['user_id'].nunique().mean()
avg_mau = avg_mau.round(5)
avg_mau

np.float64(102.58333)

In [61]:
new_df['week'] = new_df['entry_date'].dt.to_period('W')
avg_wau = new_df.groupby('week')['user_id'].nunique().mean()
avg_wau = avg_wau.round(5)
avg_wau

np.float64(89.86792)

,user_id,registration_date,entry_date,date_diff,month,week
0,1,2021-01-01,2021-01-01,0,1,2020-12-28/2021-01-03
1,1,2021-01-01,2021-01-03,2,1,2020-12-28/2021-01-03
2,1,2021-01-01,2021-01-04,3,1,2021-01-04/2021-01-10
3,1,2021-01-01,2021-01-04,3,1,2021-01-04/2021-01-10
4,1,2021-01-01,2021-01-05,4,1,2021-01-04/2021-01-10
...,...,...,...,...,...,...
20729,1000,2021-12-07,2021-12-26,19,12,2021-12-20/2021-12-26
20730,1000,2021-12-07,2021-12-28,21,12,2021-12-27/2022-01-02
20731,1000,2021-12-07,2021-12-29,22,12,2021-12-27/2022-01-02
20732,1000,2021-12-07,2021-12-30,23,12,2021-12-27/2022-01-02


In [40]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[4])

try:
    assert dec_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [41]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[5])

try:
    assert dec_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [42]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[6])

try:
    assert dec_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau усредненные**

Сохраните результат в переменные `avg_mau`, `avg_wau`, `avg_dau` соответственно

**Примечание:** результаты округлите до 5 знаков после запятой

In [56]:
# Ваше решение
new_df['month'] = new_df['entry_date'].dt.month
avg_mau = new_df.groupby('month')['user_id'].nunique().mean()
avg_mau = avg_mau.round(5)

avg_mau

np.float64(102.58333)

In [55]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[7])

try:
    assert avg_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [62]:
# Ваше решение
new_df['week'] = new_df['entry_date'].dt.to_period('W')
avg_wau = new_df.groupby('week')['user_id'].nunique().mean()
avg_wau = avg_wau.round(5)

avg_wau

np.float64(89.86792)

In [63]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[8])

try:
    assert avg_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [57]:
# Ваше решение
avg_dau = new_df.groupby('entry_date')['user_id'].nunique().mean()
avg_dau = avg_dau.round(5)

avg_dau

np.float64(40.5589)

In [51]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[9])

try:
    assert avg_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
